# Template fNIRS pipeline

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

#requires ipympl packagez
%matplotlib ipympl 

qt.qpa.xcb: QXcbConnection: XCB error: 148 (Unknown), sequence: 191, resource id: 0, major code: 141 (Unknown), minor code: 20


In [2]:
# import pyphysio
from pyphysio import TestData
from pyphysio import create_signal

nirs = TestData.fnirs(return_signal=True)
tapping = TestData.tapping(return_signal=True)

Using dask. Scheduler: threads
Please cite:
Bizzego et al. (2019) 'pyphysio: A physiological signal processing library for data science approaches in physiology', SoftwareX


In [3]:
nirs = nirs.p.reset_times(0)
plt.figure()
nirs.p.plot()

tapping = tapping.p.reset_times(0)
plt.figure()
tapping.p.plot()

In [ ]:
from pyphysio.specialized.fnirs import plot_probe, get_ss_ls_channels, get_near_channels



id_ss, id_ls = get_ss_ls_channels(nirs)
print(id_ss, id_ls)
near_1 = get_near_channels(nirs, ch_target=1)
print(near_1)

plot_probe(nirs)

## Signal Quality

In [ ]:
from pyphysio.specialized.fnirs._dl_sqi import SignalQualityDeepLearning
from pyphysio.segmenters import FixedSegments, fmap

In [ ]:
# SQI using Deep Learning
segmenter = FixedSegments(10, 20)
indicators = [SignalQualityDeepLearning()]

sqi = fmap(segmenter, indicators, nirs) #will take a while

In [ ]:
#manipulating variables and dimensions to extract information of interest
sqi = sqi['signal_SignalQualityDeepLearning'].sel({'is_good':1, 'component':0})
sqi = sqi.drop_vars(['label'])

In [ ]:
plt.figure()
sqi.p.plot('.-', sharey=True)

In [ ]:
ratio_min_good = 0.8 #at least 80% of the windows should have a good value
isgood_ratios = np.sum(sqi.values, axis=0)/sqi.values.shape[0]
id_good_channels = np.where(isgood_ratios >= ratio_min_good)[0]
nirs.attrs['good_channels'] = id_good_channels
nirs = nirs.sel({'channel': id_good_channels})

print("good channels: ", id_good_channels)

In [4]:
from pyphysio.specialized.fnirs import plot_probe, get_ss_ls_channels, get_near_channels, Raw2OD

nirs = Raw2OD()(nirs)

## Remove MA

In [5]:
import pyphysio.artefacts as artefacts

MA = artefacts.DetectMA(fuse='component')(nirs)
plt.figure()
MA.p.plot()

nirs['MA'] = MA #add MA info to nirs

nirs_noMA = artefacts.MARA()(nirs)
nirs_noMA = nirs_noMA.drop_vars('MA')

plt.figure()
nirs.p.plot(sharey=False)
nirs_noMA.p.plot(sharey=False)

#remove MA with wavelet
nirs_wav = artefacts.WaveletFilter()(nirs_noMA)

plt.figure()
nirs_noMA.p.plot(sharey=False)
nirs_wav.p.plot(sharey=False)

## Convert to Oxy/DeoxyHb

In [6]:
from pyphysio.specialized.fnirs import OD2Oxy

hb = OD2Oxy()(nirs_noMA)

plt.figure()
hb.p.plot(sharey=False)

## Remove physio noise

In [7]:
#%% remove physio noise
from pyphysio.specialized.fnirs import PCAFilter

pcafilt = PCAFilter()
hb_nophysio = pcafilt(hb)

plt.figure()
hb.p.plot(sharey=False)
hb_nophysio.p.plot(sharey=False)

## Compute clusters

In [8]:
from pyphysio.specialized.fnirs import ComputeClusters

cluster_compute = ComputeClusters(clusters=[[0,1,2], [4,5,6], [7,8,9]])
clusters = cluster_compute(hb_nophysio)

## Other filters?

In [9]:
import pyphysio.filters as filters

clusters_f = filters.IIRFilter(fp = [0.01, 0.2], btype='bandpass')(clusters)

In [10]:
plt.figure()
clusters_f.p.plot(ncols=1)

In [11]:
clusters_f.sel({'channel':[0], 'component':0}).p.plot()

qt.qpa.xcb: QXcbConnection: XCB error: 3 (BadWindow), sequence: 1842, resource id: 10540274, major code: 40 (TranslateCoords), minor code: 0


## First-level GLM analysis

In [ ]:
# convert tapping signal to event matrix
tapping_values = tapping.values
idx_onset = np.where(np.diff(tapping_values)>0)[0] - 1
t_onset = tapping.p.get_times()[idx_onset]

plt.figure()
tapping.p.plot()
plt.vlines(t_onset, 0, 1, 'r')

events = pd.DataFrame({'onset': t_onset})
events['duration'] = 5
events['stim_type'] = 1

In [ ]:
display(events)

### one-subject / one-channel analysis

In [ ]:
sub = 'S001'

In [ ]:
# create design matrix
from nilearn.glm.first_level import make_first_level_design_matrix

t = nirs.p.get_times()

dm = make_first_level_design_matrix(t, events)

In [ ]:
plt.figure()
plt.imshow(dm.values, aspect='auto', interpolation='nearest')

plt.figure()
plt.plot(t, dm.values)

In [ ]:
# run GLM
from nilearn.glm.first_level import run_glm

X = dm.values

i_ch = 0
Y = np.expand_dims(clusters_f.p.get_values()[:, i_ch, 0], 1)

assert ~np.isnan(Y).any()

labels, glm_estimates = run_glm(Y, X)
thetas = glm_estimates[labels[0]].theta[:,0]
print(thetas)

### one-subject / all channels

In [ ]:
betas = []

for i_ch in range(clusters_f.sizes['channel']):
    
        Y = np.expand_dims(clusters_f.p.get_values()[:, i_ch, 0], 1)

        assert ~np.isnan(Y).any()

        labels, glm_estimates = run_glm(Y, X)
        thetas = glm_estimates[labels[0]].theta[:,0]
        
        betas.append({'subject': sub,
                      'channel': i_ch,
                      'beta': thetas[0]})
            
betas = pd.DataFrame(betas)
print(betas)

### all subjects / all channels

In [ ]:
#get list of subjects

#for each subject:
#  - process nirs signal
#  - for each channel:
#      - run first-level GLM